# 02 - Legacy Sources and Compatibility CLI

Original BESTPRED selected input layouts through numeric `source` values. The Python
port keeps those formats for auditability and regression work, while normal application
data should enter through the DataFrame API from notebook 01.

In [1]:
from pathlib import Path


def find_repo_root() -> Path:
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "packages/models/bestpred").is_dir():
            return candidate
    raise RuntimeError("Run this notebook from a Bovi repository checkout")


ROOT = find_repo_root()
PACKAGE_ROOT = ROOT / "packages/models/bestpred"
FIXTURES = PACKAGE_ROOT / "tests/fixtures"
PARAMETERS = FIXTURES / "source11_current/bestpred.par"

In [2]:
import pandas as pd

sources = pd.DataFrame(
    [
        (10, "AIPL Format 4", "format4.dat", "Implemented", "Fixed-width lactation rows"),
        (
            11,
            "Testing-plan demo",
            "DCRexample.txt",
            "Implemented",
            "Plans are simulated into test days",
        ),
        (12, "USDA master records", "input.dcr", "Not implemented", "Enum only; no Python parser"),
        (
            13,
            "Research RIP records",
            "site-specific",
            "Out of scope",
            "Not exposed by the Python enum",
        ),
        (14, "DRMS/PCDART", "PCDART text", "Implemented", "Can also write PCDART output"),
        (15, "Format 4 + means", "format4.dat + .means", "Implemented", "Adds 305-day means"),
        (
            24,
            "List of source-14 files",
            "pcdart_files.txt",
            "Implemented",
            "Flattens multiple DRMS files",
        ),
    ],
    columns=["Source", "Meaning", "Input", "Python status", "Notes"],
)
sources

,Source,Meaning,Input,Python status,Notes
0,10,AIPL Format 4,format4.dat,Implemented,Fixed-width lactation rows
1,11,Testing-plan demo,DCRexample.txt,Implemented,Plans are simulated into test days
2,12,USDA master records,input.dcr,Not implemented,Enum only; no Python parser
3,13,Research RIP records,site-specific,Out of scope,Not exposed by the Python enum
4,14,DRMS/PCDART,PCDART text,Implemented,Can also write PCDART output
5,15,Format 4 + means,format4.dat + .means,Implemented,Adds 305-day means
6,24,List of source-14 files,pcdart_files.txt,Implemented,Flattens multiple DRMS files


## Inspect the raw shapes

In [3]:
source10_path = FIXTURES / "source10_current/format4.dat"
source11_path = FIXTURES / "source11_current/DCRexample.txt"
source14_path = FIXTURES / "source14_current/test241.txt"

raw_samples = {
    "source 10": source10_path.read_text(errors="replace").splitlines()[0][:120],
    "source 11": next(
        line
        for line in source11_path.read_text().splitlines()
        if line.strip() and not line.startswith("_")
    ),
    "source 14": source14_path.read_text(errors="replace").splitlines()[1][:120],
}
pd.Series(raw_samples, name="First representative line").to_frame()

,First representative line
source 10,0FHOUSA00093WNM4798HOUSA000001935264 ...
source 11,1 2 2 2 1 30 1...
source 14,100000011000001H02002080200001200510231 302 ...


## Parse every supported fixture into the common typed boundary

In [4]:
from bestpred.core.source11 import simulate_source11_records
from bestpred.io.parameters import read_parameters
from bestpred.io.source10 import read_source10_records
from bestpred.io.source11 import read_source11_examples
from bestpred.io.source14 import read_source14_records, read_source24_records
from bestpred.io.source15 import read_source15_records

parameters = read_parameters(PARAMETERS)
parsed = {
    10: read_source10_records(source10_path),
    11: simulate_source11_records(read_source11_examples(source11_path), parameters),
    14: read_source14_records(source14_path),
    15: read_source15_records(
        FIXTURES / "source15_current/format4.dat",
        FIXTURES / "source15_current/format4.means",
    ),
    24: read_source24_records(FIXTURES / "source24_current/pcdart_files.txt"),
}
pd.DataFrame(
    [
        {
            "Source": source,
            "Records": len(records),
            "Records with test days": sum(bool(record.segments) for record in records),
            "First cow": records[0].cow_id,
        }
        for source, records in parsed.items()
    ]
)

,Source,Records,Records with test days,First cow
0,10,2,1,HOUSA00093WNM4798
1,11,43,43,HOUSA.EX.COW.0001
2,14,3,2,H0 1000001
3,15,2,1,HOUSA00093WNM4798
4,24,6,4,H0 1000001


Some counts intentionally include current-Fortran artifacts. Sources 10 and 15 preserve
a header/flush row, and source 14 preserves an EOF row. These are compatibility behavior,
not recommended domain modeling. See `docs/reference/port/fortran-quirks.md` before
removing or interpreting such rows.

## Run the compatibility CLI without writing into the checkout

In [5]:
import subprocess
import sys
import tempfile

from bestpred.io.dcr import read_dcr_results

with tempfile.TemporaryDirectory(prefix="bestpred-notebook-") as temp_dir:
    output_path = Path(temp_dir) / "results_v2.dcr"
    completed = subprocess.run(
        [
            sys.executable,
            "-m",
            "bestpred.cli",
            "run",
            "--source",
            "10",
            "--input",
            str(source10_path),
            "--par",
            str(FIXTURES / "source10_current/bestpred.par"),
            "--output",
            str(output_path),
        ],
        check=True,
        capture_output=True,
        text=True,
    )
    cli_rows = read_dcr_results(output_path)

pd.DataFrame(
    {
        "Return code": [completed.returncode],
        "DCR rows": [len(cli_rows)],
        "Numeric fields per row": [len(cli_rows[0].numeric_values)],
    }
)

,Return code,DCR rows,Numeric fields per row
0,0,2,43


The CLI requires `--source`, `--input`, `--par`, and `--output`. Sources 14/24 may also
receive `--pcdart-output`; source 15 requires a sibling `.means` file. The parameter file
controls model length, traits, interpolation and input/output units. Use source formats
for legacy exchange and oracle tests, not as the first choice for a new service contract.